<a href="https://colab.research.google.com/github/DeisyUdec/DETECCI-N-DEL-CIBERBULLYING-CON-LA-IMPLEMENTACI-N-IA/blob/main/proyecto_ciberbullying.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas scikit-learn gradio

In [3]:
import pandas as pd
import re
import unicodedata

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Cargar el dataset con el delimitador correcto
df = pd.read_csv("/content/CorpusBullying (1).csv", sep=";")

# Renombrar columnas para que coincidan con el resto del código
df = df.rename(columns={
    "SpanishTweet": "texto",
    "Label": "etiqueta"
})

# Revisar los datos
print(df.head())
print(df["etiqueta"].value_counts())

# Eliminar registros vacíos
df = df.dropna(subset=["texto", "etiqueta"])

# Preparar los datos
X = df["texto"].astype(str)
y = df["etiqueta"].astype(int)

# Dividir los datos
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Crear el modelo
modelo = Pipeline([
    ("vectorizador", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        max_features=20000
    )),
    ("clasificador", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

# Entrenar
modelo.fit(X_train, y_train)

# Evaluar
predicciones = modelo.predict(X_test)

print("Exactitud:", accuracy_score(y_test, predicciones))
print(classification_report(y_test, predicciones))

   ID                                              texto  etiqueta
0   1  venga t o regalad simo aparece ellsworth y dis...       0.0
1   2  que no te quiere porque no sos su tipo y eres ...       1.0
2   3  tres tipo de gente que repugno gente mentirosa...       0.0
3   4  rt dejulio lo dije hace un a o y lo repito hoy...       0.0
4   5  jajajajajaja jajajajajaja pega extraaa jajajaj...       1.0
etiqueta
0.0    34214
1.0     2267
Name: count, dtype: int64
Exactitud: 0.9543648074551185
              precision    recall  f1-score   support

           0       1.00      0.95      0.98      6844
           1       0.58      0.95      0.72       453

    accuracy                           0.95      7297
   macro avg       0.79      0.95      0.85      7297
weighted avg       0.97      0.95      0.96      7297



In [4]:
def analizar_comentario(comentario):

    if not comentario.strip():
        return "Por favor, escribe un comentario."

    # Obtener la probabilidad de posible ciberbullying
    probabilidades = modelo.predict_proba([comentario])[0]

    probabilidad_ciberbullying = probabilidades[
        list(modelo.classes_).index(1)
    ]

    # Umbral de decisión
    umbral = 0.60

    if probabilidad_ciberbullying >= umbral:

        return (
            "🚫 PUBLICACIÓN BLOQUEADA\n\n"
            "El comentario presenta posibles características "
            "de ciberbullying.\n"
            "Por favor, revisa el mensaje antes de publicarlo.\n\n"
            f"Probabilidad estimada: "
            f"{probabilidad_ciberbullying * 100:.2f}%"
        )

    else:

        return (
            "✅ PUBLICACIÓN PERMITIDA\n\n"
            "No se detectaron suficientes características "
            "de ciberbullying en el comentario.\n\n"
            f"Probabilidad estimada de ciberbullying: "
            f"{probabilidad_ciberbullying * 100:.2f}%"
        )

In [9]:
import gradio as gr

# Diseño de pantalla oscura
css_personalizado = """
body, .gradio-container {
    background: #000000 !important;
    color: #ffffff !important;
}

.gradio-container {
    max-width: 900px !important;
    margin: auto !important;
}
.gradio-container .image-container,
.gradio-container [data-testid="image"] {
    background: #808080 !important;
    border-radius: 12px !important;
    padding: 15px !important;
}

.block, .form, .panel {
    background: #161616 !important;
    border: 1px solid #333333 !important;
    color: #ffffff !important;
    border-radius: 12px !important;
}

label, p, h1, h2, span {
    color: #ffffff !important;
}

textarea, input {
    background: #242424 !important;
    color: #ffffff !important;
    border: 1px solid #444444 !important;
    border-radius: 10px !important;
}

button {
    background: #1976d2 !important;
    color: white !important;
    border-radius: 12px !important;
    border: none !important;
    padding: 10px 20px !important;
}
"""

# Crear la interfaz
with gr.Blocks(
    theme=gr.themes.Base(),
    css=css_personalizado
) as interfaz_completa:

    gr.Markdown("# Sistema de detección de ciberbullying")

    gr.Image(
        value="/content/imagen de publicacion.webp",
        show_label=False,
        interactive=False,
        height=250,container=True
    )

    comentario = gr.Textbox(
        lines=4,
        label="Escribe tu comentario",
        placeholder="Ingresa aquí el mensaje que deseas publicar..."
    )

    boton = gr.Button("Analizar comentario")

    resultado = gr.Textbox(
        label="Resultado del análisis",
        lines=6
    )

    boton.click(
        fn=analizar_comentario,
        inputs=comentario,
        outputs=resultado
    )

interfaz_completa.launch()

/tmp/ipykernel_676/1566793050.py:49: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://349ad1a156e8764941.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
!pip install -q -U pandas numpy scikit-learn tensorflow transformers datasets accelerate

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Crear modelo
modelo_logistico = Pipeline([
    ("vectorizador", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        max_features=20000
    )),
    ("clasificador", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

# Entrenar
modelo_logistico.fit(X_train, y_train)

print("Regresión logística entrenada correctamente")

In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.layers import (
    TextVectorization, Embedding, LSTM,
    Dense, Dropout
)
from tensorflow.keras.models import Sequential
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split

# Crear el conjunto de validación que falta a partir del conjunto de entrenamiento
X_train_actual, X_val, y_train_actual, y_val = train_test_split(
    X_train, y_train,
    test_size=0.10,
    random_state=42,
    stratify=y_train
)

# Preparar los textos
vectorizador = TextVectorization(
    max_tokens=20000,
    output_mode="int",
    output_sequence_length=120
)

vectorizador.adapt(
    tf.data.Dataset.from_tensor_slices(
        X_train_actual.to_numpy()
    ).batch(64)
)

# Convertir textos en secuencias
X_train_lstm = vectorizador(
    tf.constant(X_train_actual.to_numpy())
)
X_val_lstm = vectorizador(
    tf.constant(X_val.to_numpy())
)
X_test_lstm = vectorizador(
    tf.constant(X_test.to_numpy())
)

# Crear modelo LSTM
modelo_lstm = Sequential([
    Embedding(20000, 128),
    LSTM(64),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid")
])

modelo_lstm.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Balancear las clases durante el entrenamiento
clases = np.unique(y_train_actual)

pesos = compute_class_weight(
    class_weight="balanced",
    classes=clases,
    y=y_train_actual
)

pesos_clases = dict(zip(clases, pesos))

# Entrenar utilizando los nuevos conjuntos divididos
historial = modelo_lstm.fit(
    X_train_lstm,
    y_train_actual,
    validation_data=(X_val_lstm, y_val),
    epochs=5,
    batch_size=64,
    class_weight=pesos_clases,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=2,
            restore_best_weights=True
        )
    ]
)

print("Modelo LSTM entrenado correctamente")

In [ ]:
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Nombre del modelo BETO
nombre_modelo = "dccuchile/bert-base-spanish-wwm-cased"

# Cargar tokenizador
tokenizer = AutoTokenizer.from_pretrained(nombre_modelo)

# Preparar conjuntos de datos
datos_train = Dataset.from_dict({
    "text": X_train.tolist(),
    "label": y_train.tolist()
})

datos_val = Dataset.from_dict({
    "text": X_val.tolist(),
    "label": y_val.tolist()
})

datos_test = Dataset.from_dict({
    "text": X_test.tolist(),
    "label": y_test.tolist()
})

# Tokenizar comentarios
def tokenizar(datos):
    return tokenizer(
        datos["text"],
        truncation=True,
        max_length=128
    )

train_tokenizado = datos_train.map(
    tokenizar, batched=True, remove_columns=["text"]
)

val_tokenizado = datos_val.map(
    tokenizar, batched=True, remove_columns=["text"]
)

test_tokenizado = datos_test.map(
    tokenizar, batched=True, remove_columns=["text"]
)

# Cargar BETO para clasificación
modelo_beto = AutoModelForSequenceClassification.from_pretrained(
    nombre_modelo,
    num_labels=2
)

# Métricas
def calcular_metricas(eval_pred):
    logits, etiquetas = eval_pred
    predicciones = np.argmax(logits, axis=1)

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            etiquetas,
            predicciones,
            average="binary",
            zero_division=0
        )
    )

    return {
        "accuracy": accuracy_score(etiquetas, predicciones),
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

# Configuración del entrenamiento
argumentos = TrainingArguments(
    output_dir="./resultados_beto",
    num_train_epochs=2,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=1,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

# Entrenador
entrenador = Trainer(
    model=modelo_beto,
    args=argumentos,
    train_dataset=train_tokenizado,
    eval_dataset=val_tokenizado,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=calcular_metricas
)

# Entrenar BETO
entrenador.train()

print("Modelo BETO entrenado correctamente")

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Predicciones de Regresión logística
pred_logistica = modelo_logistico.predict(X_test)

# Predicciones de LSTM
prob_lstm = modelo_lstm.predict(
    X_test_lstm
).ravel()

pred_lstm = (prob_lstm >= 0.5).astype(int)

# Predicciones de BETO
resultado_beto = entrenador.predict(test_tokenizado)

pred_beto = np.argmax(
    resultado_beto.predictions,
    axis=1
)

# Función para calcular métricas
def evaluar(nombre, reales, predicciones):
    return {
        "Modelo": nombre,
        "Accuracy": accuracy_score(reales, predicciones),
        "Precision": precision_score(
            reales, predicciones, zero_division=0
        ),
        "Recall": recall_score(
            reales, predicciones, zero_division=0
        ),
        "F1-score": f1_score(
            reales, predicciones, zero_division=0
        )
    }

# Comparar modelos
resultados = pd.DataFrame([
    evaluar("Regresión logística", y_test, pred_logistica),
    evaluar("LSTM", y_test, pred_lstm),
    evaluar("BETO", y_test, pred_beto)
])

print(resultados.round(4))

# Identificar el modelo con mejor F1-score
mejor_modelo = resultados.loc[
    resultados["F1-score"].idxmax()
]

print("\nMejor modelo:")
print(mejor_modelo)

In [ ]:
# Seleccionar el mejor modelo
nombre_mejor = mejor_modelo["Modelo"]

def analizar_comentario(comentario):

    if not comentario.strip():
        return "Por favor, escribe un comentario."

    # Obtener probabilidad según el modelo seleccionado
    if nombre_mejor == "Regresión logística":

        probabilidad = modelo_logistico.predict_proba(
            [comentario]
        )[0][1]

    elif nombre_mejor == "LSTM":

        secuencia = vectorizador(
            tf.constant([comentario])
        )

        probabilidad = float(
            modelo_lstm.predict(secuencia, verbose=0)[0][0]
        )

    else:

        entrada = tokenizer(
            comentario,
            return_tensors="pt",
            truncation=True,
            max_length=128
        )

        dispositivo = next(
            entrenador.model.parameters()
        ).device

        entrada = {
            k: v.to(dispositivo)
            for k, v in entrada.items()
        }

        entrenador.model.eval()

        with torch.no_grad():
            salida = entrenador.model(**entrada)

        probabilidad = torch.softmax(
            salida.logits, dim=1
        )[0][1].item()

    # Decisión
    if probabilidad >= 0.5:
        return (
            "🚫 PUBLICACIÓN BLOQUEADA\n"
            "Se detectaron posibles características de ciberbullying.\n"
            "Se recomienda revisar el comentario.\n"
            f"Probabilidad estimada: {probabilidad*100:.2f}%"
        )

    else:
        return (
            "✅ PUBLICACIÓN PERMITIDA\n"
            "No se detectaron suficientes características de ciberbullying.\n"
            f"Probabilidad estimada: {probabilidad*100:.2f}%"
        )